# Momentum + Roll-Impact IS/OS Test - 1h Candles

Hypothesis: `vol_adj_momentum_24` remains the primary alpha candidate. `roll_impact` is no longer treated as a direct alpha overlay; it is tested only as a risk/execution-quality modifier that may reduce bad fills, stop hits, or left-tail exposure.

Default target: 24h forward return from 1h candles. Current split intent: 3 months in-sample, then 2 months out-of-sample, matching the fixed V1 ridge deployment window.

This notebook is reusable for the same policy grid, but current production comparisons should use `data/features/live_1h_features_30m.csv`.


## Data Note

This notebook now expects the canonical 30-month 1h feature file `data/features/live_1h_features_30m.csv`.

`ALLOW_SHORT_SAMPLE_FALLBACK = False`, so the notebook will fail instead of silently falling back if the requested 3m IS / 2m OS split is not covered. Old short-sample notebook outputs were cleared because they are not valid evidence for the current V1/V2 refit hypothesis.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import HTML, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = next(parent for parent in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents) if (parent / "bot").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bot.forward_ic import add_forward_return  # noqa: E402

FEATURE_PATH = REPO_ROOT / "data/features/live_1h_features_30m.csv"
OUTPUT_DIR = REPO_ROOT / "notebooks/microstructure"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 24
IS_MONTHS = 3
OS_MONTHS = 2
ALLOW_SHORT_SAMPLE_FALLBACK = False
SHORT_SAMPLE_IS_FRACTION = 0.70
COST_BPS = 20
TOP_KS = (3, 5)
RISK_CUTOFFS = (0.50, 2 / 3, 0.75, 0.85)
ALPHA_SPECS = (
    {
        "name": "vol_adj_momentum_24",
        "feature_col": "vol_adj_momentum_24",
        "ascending": False,
    },
)
RISK_SPECS = (
    {
        "name": "roll_impact",
        "risk_col": "roll_impact",
        "ascending": True,
    },
)
POLICIES = ("pure_momentum", "gate_high_risk", "scaled_by_low_risk")
MIN_ASSETS_PER_TIMESTAMP = 8


In [ ]:
def normalize_time(frame: pd.DataFrame, time_col: str = "open_time") -> pd.DataFrame:
    out = frame.copy()
    if pd.api.types.is_numeric_dtype(out[time_col]):
        out["timestamp"] = pd.to_datetime(out[time_col], unit="ms", utc=True)
    else:
        out["timestamp"] = pd.to_datetime(out[time_col], utc=True)
    return out


def load_feature_frame() -> pd.DataFrame:
    df = pd.read_csv(FEATURE_PATH)
    df = normalize_time(df)
    target_col = f"forward_return_{HORIZON}"
    if target_col not in df.columns:
        df = add_forward_return(
            df,
            horizon=HORIZON,
            price_col="close",
            target_col=target_col,
            group_col="pair",
            sort_col="open_time",
        )
    df = df.sort_values(["pair", "timestamp"]).reset_index(drop=True)
    return df


def assign_split(
    frame: pd.DataFrame,
    is_months: int = IS_MONTHS,
    os_months: int = OS_MONTHS,
) -> tuple[pd.DataFrame, dict]:
    out = frame.copy()
    unique_times = pd.Series(out["timestamp"].dropna().sort_values().unique())
    start = pd.Timestamp(unique_times.iloc[0])
    end = pd.Timestamp(unique_times.iloc[-1])
    target_split_at = start + pd.DateOffset(months=is_months)
    target_end = target_split_at + pd.DateOffset(months=os_months)
    coverage_ok = end >= target_end

    if coverage_ok:
        split_at = target_split_at
        os_end = target_end
        split_mode = f"{is_months}m_is_{os_months}m_os"
        out["sample"] = "unused"
        out.loc[out["timestamp"] < split_at, "sample"] = "IS"
        out.loc[(out["timestamp"] >= split_at) & (out["timestamp"] < os_end), "sample"] = "OS"
    elif ALLOW_SHORT_SAMPLE_FALLBACK:
        split_index = max(1, min(len(unique_times) - 1, int(len(unique_times) * SHORT_SAMPLE_IS_FRACTION)))
        split_at = pd.Timestamp(unique_times.iloc[split_index])
        os_end = end
        split_mode = f"fallback_{SHORT_SAMPLE_IS_FRACTION:.0%}_is_short_sample"
        out["sample"] = "OS"
        out.loc[out["timestamp"] < split_at, "sample"] = "IS"
    else:
        raise ValueError(
            f"Need data through {target_end} for a true {is_months}m/{os_months}m split; "
            f"available ends at {end}."
        )

    metadata = {
        "coverage_ok": bool(coverage_ok),
        "split_mode": split_mode,
        "available_start": str(start),
        "available_end": str(end),
        "available_days": float((end - start).total_seconds() / 86400),
        "target_split_at": str(target_split_at),
        "target_end": str(target_end),
        "actual_split_at": str(split_at),
        "actual_os_end": str(os_end),
        "is_months": is_months,
        "os_months": os_months,
    }
    return out[out["sample"].isin(["IS", "OS"])].copy(), metadata


def add_risk_ranks(frame: pd.DataFrame, risk_specs: tuple[dict, ...]) -> pd.DataFrame:
    out = frame.copy()
    for spec in risk_specs:
        rank_col = f"{spec['name']}_risk_rank"
        out[rank_col] = out.groupby("open_time", observed=True)[spec["risk_col"]].rank(
            ascending=spec.get("ascending", True),
            pct=True,
        )
    return out


def top_k_by_score(frame: pd.DataFrame, score_col: str, top_k: int) -> pd.DataFrame:
    ranked = frame.dropna(subset=[score_col]).copy()
    ranked["score_rank"] = ranked.groupby("open_time", observed=True)[score_col].rank(
        ascending=False,
        method="first",
    )
    return ranked[ranked["score_rank"] <= top_k].copy()


In [ ]:
def policy_candidates(
    frame: pd.DataFrame,
    alpha_spec: dict,
    risk_spec: dict,
    policy: str,
    top_k: int,
    risk_cutoff: float,
    target_col: str,
) -> pd.DataFrame:
    alpha_col = alpha_spec["feature_col"]
    risk_rank_col = f"{risk_spec['name']}_risk_rank"
    candidate_cols = [
        "open_time",
        "timestamp",
        "sample",
        "pair",
        alpha_col,
        risk_rank_col,
        target_col,
    ]
    candidates = frame[candidate_cols].dropna(subset=[alpha_col, risk_rank_col, target_col]).copy()
    enough_assets = candidates.groupby("open_time", observed=True)["pair"].transform("count")
    candidates = candidates[enough_assets >= MIN_ASSETS_PER_TIMESTAMP]

    if policy == "pure_momentum":
        score_col = alpha_col
    elif policy == "gate_high_risk":
        candidates = candidates[candidates[risk_rank_col] <= risk_cutoff].copy()
        score_col = alpha_col
    elif policy == "scaled_by_low_risk":
        score_col = "risk_scaled_score"
        candidates[score_col] = candidates[alpha_col] * (1 - candidates[risk_rank_col])
    else:
        raise ValueError(f"Unknown policy: {policy}")

    selected = top_k_by_score(candidates, score_col, top_k)
    selected["alpha"] = alpha_spec["name"]
    selected["risk_filter"] = risk_spec["name"]
    selected["policy"] = policy
    selected["top_k"] = top_k
    selected["risk_cutoff"] = risk_cutoff
    selected["selected_return"] = selected[target_col]
    return selected


def evaluate_grid(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_col = f"forward_return_{HORIZON}"
    selected_parts = []
    for alpha_spec in ALPHA_SPECS:
        for risk_spec in RISK_SPECS:
            for top_k in TOP_KS:
                for policy in POLICIES:
                    cutoffs = RISK_CUTOFFS if policy == "gate_high_risk" else (1.0,)
                    for cutoff in cutoffs:
                        selected_parts.append(
                            policy_candidates(
                                frame,
                                alpha_spec,
                                risk_spec,
                                policy,
                                top_k,
                                cutoff,
                                target_col,
                            )
                        )

    selections = pd.concat(selected_parts, ignore_index=True)
    group_cols = ["sample", "alpha", "risk_filter", "policy", "top_k", "risk_cutoff"]
    period_returns = selections.groupby([*group_cols, "open_time", "timestamp"], observed=True).agg(
        portfolio_return=("selected_return", "mean"),
        selected_count=("pair", "count"),
    )
    period_returns = period_returns.reset_index()
    period_returns["net_return"] = period_returns["portfolio_return"] - COST_BPS / 10_000

    summaries = []
    for keys, group in period_returns.groupby(group_cols, observed=True):
        key_data = dict(zip(group_cols, keys, strict=True))
        std = group["net_return"].std()
        mean_net = group["net_return"].mean()
        summaries.append(
            {
                **key_data,
                "periods": len(group),
                "mean_return": group["portfolio_return"].mean(),
                "mean_net_return": mean_net,
                "median_net_return": group["net_return"].median(),
                "std_net_return": std,
                "sharpe_like": mean_net / std if std and not pd.isna(std) else float("nan"),
                "hit_rate": (group["portfolio_return"] > 0).mean(),
                "net_hit_rate": (group["net_return"] > 0).mean(),
                "q05_net_return": group["net_return"].quantile(0.05),
                "q95_net_return": group["net_return"].quantile(0.95),
                "avg_selected_count": group["selected_count"].mean(),
            }
        )
    return pd.DataFrame(summaries), period_returns


def select_is_winner(summary: pd.DataFrame) -> pd.Series:
    is_summary = summary[summary["sample"].eq("IS")].copy()
    candidates = is_summary[is_summary["policy"].ne("pure_momentum")].copy()
    if candidates.empty:
        raise ValueError("No non-pure momentum candidates available for IS selection")
    candidates = candidates.sort_values(
        ["mean_net_return", "sharpe_like", "periods"],
        ascending=[False, False, False],
    )
    return candidates.iloc[0]


def compare_selected_to_baseline(summary: pd.DataFrame, selected: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    baseline = summary[
        summary["alpha"].eq(selected["alpha"])
        & summary["risk_filter"].eq(selected["risk_filter"])
        & summary["top_k"].eq(selected["top_k"])
        & summary["policy"].eq("pure_momentum")
    ].copy()
    chosen = summary[
        summary["alpha"].eq(selected["alpha"])
        & summary["risk_filter"].eq(selected["risk_filter"])
        & summary["top_k"].eq(selected["top_k"])
        & summary["policy"].eq(selected["policy"])
        & summary["risk_cutoff"].eq(selected["risk_cutoff"])
    ].copy()
    baseline["variant"] = "baseline_pure_momentum"
    chosen["variant"] = "selected_filter"
    combined = pd.concat([baseline, chosen], ignore_index=True)
    pivot = combined.pivot_table(
        index="sample",
        columns="variant",
        values="mean_net_return",
        aggfunc="first",
    ).reset_index()
    if {"baseline_pure_momentum", "selected_filter"}.issubset(pivot.columns):
        pivot["net_improvement"] = pivot["selected_filter"] - pivot["baseline_pure_momentum"]
    return combined, pivot


In [ ]:
def svg_bar(
    df: pd.DataFrame,
    title: str,
    label_col: str,
    value_col: str,
    width: int = 900,
    left: int = 230,
    row_h: int = 28,
) -> str:
    rows = df[[label_col, value_col]].dropna().copy().sort_values(value_col)
    height = 54 + row_h * len(rows)
    plot_w = width - left - 90
    max_abs = max(float(rows[value_col].abs().max()), 0.001)
    zero_x = left + plot_w / 2
    scale = (plot_w / 2) / max_abs
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">'
    ]
    parts.append('<rect width="100%" height="100%" fill="#fff"/>')
    parts.append(f'<text x="12" y="24" font-family="Arial" font-size="16" font-weight="700">{title}</text>')
    parts.append(f'<line x1="{zero_x:.1f}" y1="38" x2="{zero_x:.1f}" y2="{height - 10}" stroke="#555"/>')
    for i, row in enumerate(rows.itertuples(index=False)):
        label = str(getattr(row, label_col)).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        val = float(getattr(row, value_col))
        y = 46 + i * row_h
        bar_w = abs(val) * scale
        x = zero_x if val >= 0 else zero_x - bar_w
        color = "#2f7d32" if val >= 0 else "#b23a3a"
        parts.append(f'<text x="12" y="{y + 15}" font-family="Arial" font-size="12">{label}</text>')
        parts.append(f'<rect x="{x:.1f}" y="{y}" width="{bar_w:.1f}" height="18" fill="{color}" opacity="0.84"/>')
        tx = zero_x + bar_w + 6 if val >= 0 else zero_x - bar_w - 70
        parts.append(f'<text x="{tx:.1f}" y="{y + 15}" font-family="Arial" font-size="12">{val:+.4f}</text>')
    parts.append("</svg>")
    return "".join(parts)


In [ ]:
features = load_feature_frame()
features, split_metadata = assign_split(features)
features = add_risk_ranks(features, RISK_SPECS)
summary, period_returns = evaluate_grid(features)
selected = select_is_winner(summary)
selected_comparison, selected_pivot = compare_selected_to_baseline(summary, selected)

summary.to_csv(OUTPUT_DIR / "momentum_roll_impact_is_os_1h_policy_summary.csv", index=False)
period_returns.to_csv(OUTPUT_DIR / "momentum_roll_impact_is_os_1h_period_returns.csv", index=False)
selected_comparison.to_csv(OUTPUT_DIR / "momentum_roll_impact_is_os_1h_selected_comparison.csv", index=False)
selected_pivot.to_csv(OUTPUT_DIR / "momentum_roll_impact_is_os_1h_selected_pivot.csv", index=False)

metadata = {
    **split_metadata,
    "source_features": str(FEATURE_PATH),
    "horizon": HORIZON,
    "cost_bps": COST_BPS,
    "top_ks": list(TOP_KS),
    "risk_cutoffs": [float(x) for x in RISK_CUTOFFS],
    "alpha_specs": list(ALPHA_SPECS),
    "risk_specs": list(RISK_SPECS),
    "policies": list(POLICIES),
    "rows": int(len(features)),
    "pairs": int(features["pair"].nunique()),
    "selected_policy": selected.to_dict(),
}
(OUTPUT_DIR / "momentum_roll_impact_is_os_1h_metadata.json").write_text(
    json.dumps(metadata, indent=2, sort_keys=True, default=str),
    encoding="utf-8",
)
metadata


## Split Audit

Do not trust OS results unless `coverage_ok` in metadata is `True`.


In [ ]:
pd.DataFrame(
    [
        {
            "sample": sample,
            "start": str(group["timestamp"].min()),
            "end": str(group["timestamp"].max()),
            "bars": group["open_time"].nunique(),
            "rows": len(group),
            "pairs": group["pair"].nunique(),
        }
        for sample, group in features.groupby("sample", observed=True)
    ]
)


## In-Sample Grid

This is where the risk-filter variant is selected. We select only from non-pure-momentum policies, then freeze that choice before looking at OS.


In [ ]:
summary[summary["sample"].eq("IS")].sort_values("mean_net_return", ascending=False).head(12)


## Out-of-Sample Grid

This is not used for selection. It is the holdout check for the frozen IS-selected policy.


In [ ]:
summary[summary["sample"].eq("OS")].sort_values("mean_net_return", ascending=False).head(12)


## Selected Filter vs Baseline

The comparison uses the same alpha and top-K for both rows. The only difference is whether the roll-impact filter selected in IS is applied.


In [ ]:
selected_comparison.sort_values(["sample", "variant"])


In [ ]:
selected_pivot.sort_values("sample")


In [ ]:
plot_frame = selected_comparison.copy()
plot_frame["label"] = plot_frame.apply(lambda r: f"{r['sample']} {r['variant']}", axis=1)
display(HTML(svg_bar(plot_frame, "Selected IS filter vs pure momentum", "label", "mean_net_return")))


## How To Read This

        - If the selected filter beats pure momentum in IS but fails in OS, the filter likely overfit this sample.
        - If it improves OS `mean_net_return` and does not rely on a much lower `periods` count, the hypothesis gets stronger.
        - If `scaled_by_low_risk` beats hard gating, roll impact may be better as position sizing/exposure control than as a binary entry filter.
        - If the selected policy changes when more alphas or longer data are added, treat this as model selection and rerun the IS/OS protocol.
